# BPIC15 Digital Shadow Evolution — Concrete Notebook Plan

This notebook is the **implementation roadmap** for building an evolving digital shadow of bureaucratic workflows using real data splits and RL.

**Research Framework**: Based on **Kritzinger et al. (2018)** Digital Shadow definition and **Tao et al. (2018)** digital twin lifecycle.

---

## 0) Scope, assumptions, and research contribution

**Research Question:** 
*How can an evolving digital shadow (trained on increasingly complete historical data) reveal changing process bottlenecks and inefficiencies through RL policy adaptation?*

**Primary inputs**
- `./dataset/BPIC15_Municipality{1..5}.jsonocel` (all cases, 100% data)
- Temporal split: 70% (early period) | 30% (new period)

**Primary outputs (write to `./output`)**

**Digital Shadow v1 (70% data):**
- `dt_v1/case_step_features.parquet`
- `dt_v1/transition_stats.csv`
- `dt_v1/duration_stats.csv`
- `dt_v1/sim_calibration.json`
- `dt_v1/ppo_checkpoint/`
- `dt_v1/actions_sequences.csv`

**Digital Shadow v2 (100% data - evolved):**
- `dt_v2/case_step_features.parquet`
- `dt_v2/transition_stats.csv`
- `dt_v2/duration_stats.csv`
- `dt_v2/sim_calibration.json`
- `dt_v2/ppo_checkpoint/`
- `dt_v2/actions_sequences.csv`

**Evolution & Comparison:**
- `shadow_evolution_report.json` (how parameters changed)
- `policy_comparison_rules_v1_vs_v2.csv` (extracted business rules)
- `managerial_dashboard_v1_vs_v2.ipynb` (interactive recommendations)

**Acceptance criteria**
- Real vs simulated distributions close on core metrics (both versions)
- Both PPO policies are stable and converge
- Differences between Policy v1 and Policy v2 are interpretable as workflow evolution

---

## 1) Recreate and freeze graph-derived priors (from process_graph)
**Why:** Downstream feature engineering needs stable stage/order signals and branch hints.

**Cells to add**
1. Load logs and rebuild minimal shared objects (`logs`, `dfgs`, `act_ranks`)
2. Compute shared backbone transitions (intersection / high-support edges)
3. Compute branch/rare transitions (low-support or municipality-specific)
4. Persist priors to disk (`graph_priors.json`)

**Checks**
- At least one non-empty backbone set
- Rare transitions include meaningful deviations (not only noise)

---

## 2) Build the case-step feature table (one row per event-in-case)
**Why:** This is the RL state table.

**Target schema (minimum)**
- `municipality`, `case_id`, `event_id`, `timestamp`
- `activity`, `prev_activity`, `next_activity`
- `step_index`, `trace_length`, `stage_rank`
- `time_since_case_start_hours`, `time_since_prev_hours`
- `rework_count_activity`, `seen_activity_before`
- branch flags: `is_refusal_path`, `is_suspension_path`, `is_completeness_path`
- outcome helpers: `is_terminal_event`, `case_completed`

**Cells to add**
1. Build `Application -> ordered events` extractor
2. Row-expansion function per case
3. Concatenate all municipalities into one dataframe
4. Type-cleaning + null handling + sort keys
5. Save `case_step_features.parquet`

**Checks**
- No duplicate key on (`municipality`,`case_id`,`event_id`)
- Time deltas are non-negative
- `step_index` spans `0..trace_length-1` per case

---

## 3) Compute duration and branch statistics for simulator calibration
**Why:** Simulator parameters must be learned from data, not guessed.

**Cells to add**
1. Transition probabilities by municipality and global
2. Activity duration distributions (median, IQR, log-mean if skewed)
3. Branch probabilities from designated decision activities
4. Case-arrival proxy from first-event timestamps
5. Persist as `transition_stats.csv`, `duration_stats.csv`, `sim_calibration.json`

**Checks**
- Transition rows sum to ~1 per source activity
- No empty duration model for frequent activities

---

## 4) Define a small, explicit action space
**Why:** PPO requires clear, enforceable decisions.

**Initial actions**
- `continue_normal_path`
- `prioritize_case`
- `request_missing_info`
- `send_to_review`
- `escalate`
- `close_case`

**Cells to add**
1. Action dictionary and integer mapping
2. State-dependent validity rules (action mask function)
3. Export/refresh `valid_action_space.csv` with rule descriptions

**Checks**
- Every state has at least one valid action
- Invalid actions are deterministic from state features

---

## 5) Specify reward function (baseline)
Use a simple linear reward:

$$
r_t = -\alpha\,\text{delay}_t - \beta\,\text{rework}_t - \delta\,\text{invalid}_t + \gamma\,\text{completion}_t
$$

**Cells to add**
1. Parameter block (`alpha, beta, delta, gamma`)
2. Reward component function returning per-step breakdown
3. Sanity plots of reward distributions by municipality

**Checks**
- Completed cases have higher cumulative reward than stalled cases (median)
- Invalid-action penalty dominates tiny delay differences

---

## 6) Build a calibrated SimPy environment
**Why:** PPO needs many trajectories; simulator must mimic real workflow behavior.

**Cells to add**
1. Case generator using arrival-rate estimates
2. Event progression using transition probabilities + duration model
3. Hook for agent action to influence routing/priority
4. Episode termination and logging to dataframe

**Simulator output schema**
- case trace table compatible with `case_step_features` core columns
- episode summary table: duration, loops, terminal status, action counts

**Checks**
- No impossible transitions outside allowed transition table
- Episode always terminates within max step cap

---

## 7) Validate simulator against real logs (must pass before PPO)
**Comparison metrics**
1. Trace length distribution
2. Activity frequency distribution
3. Transition frequency matrix distance
4. Case duration distribution
5. Loop/rework frequency

**Cells to add**
1. Metric computation real vs simulated
2. Side-by-side plots
3. Summary table + pass/fail thresholds
4. Save `sim_validation_report.csv`

**Suggested thresholds (baseline)**
- Relative error < 20% for top activities
- Duration median error < 25%
- Loop frequency error < 20%

---

## 8) Create RL environment wrapper (Gymnasium-style)
**Observation**
- Manual feature vector from case-step row + small history signals

**Action**
- Discrete index over action space with action masking

**Episode end**
- Case completed, dropped, or max-steps reached

**Cells to add**
1. `Env` class (`reset`, `step`, `action_mask`)
2. Vectorized observation builder
3. Random-policy smoke test (10–20 episodes)

**Checks**
- Shapes/dtypes stable
- No NaN in observations/rewards
- Mask blocks invalid actions consistently

---

## 9) Train baseline PPO + Capture Trajectory Logs
**Cells to add**
1. Train/validation split strategy (e.g., municipality hold-out)
2. PPO config (small MLP, conservative learning rate)
3. Training loop + checkpoints
4. Evaluation against heuristic policy
5. **IMPORTANT**: During evaluation episodes, save (state, action, state_features) to CSV for later policy interpretation

**Minimum experiment matrix**
- Exp A: train on 1–4, test on 5
- Exp B: rotate held-out municipality
- Exp C: synthetic variant stress test

**Outputs**
- `ppo_metrics.csv`
- `ppo_checkpoint/`
- `action_trajectories.csv` ← **NEW: Timestep-level (state, action, features) from eval episodes**
- evaluation summary table

**Trajectory log format** (used by Step 11):
```
episode_id, timestep, action, activity, queue_depth, case_age_hours, rework_count, municipality, ...
```

---

---

## DIGITAL SHADOW EVOLUTION (Research Contribution)

**Research Framework**: Kritzinger et al. (2018) Digital Shadow (unidirectional sync: Physical → Digital) + Tao et al. (2018) DT lifecycle

**Core Research Question:** *How does an evolving digital shadow reveal process changes through RL policy adaptation?*

### Data Strategy: Temporal Split

**Why split data?**
- DT v1 (70% data): Baseline workflow state
- DT v2 (100% data): Evolved workflow state (with 30% new cases)
- Shows how transmission systems change as new real data arrives

**Splitting pattern:**
```python
all_data = load_all_municipalities()
sorted_data = all_data.sort_values('timestamp')
split_point = int(len(sorted_data) * 0.7)

dt_v1_data = sorted_data.iloc[:split_point]   # 70% early period
dt_v2_data = sorted_data                       # 100% all data (evolved)
```

### Execution: Steps 1-9 Run TWICE

**Digital Twin v1 (70% data):**
- Run Steps 1-7 on 70% data → Build validated simulator
- Run Steps 8-9 on DT v1 → Train PPO on v1, get Policy v1 + actions
- **OUTPUT**: `dt_v1/ppo_checkpoint/`, `dt_v1/action_sequences.csv`

**Digital Twin v2 (100% data - Evolved):**
- Run Steps 1-7 on 100% data → Recalibrate simulator (parameters evolved)
- Run Steps 8-9 on DT v2 → Train PPO on v2, get Policy v2 + actions
- **OUTPUT**: `dt_v2/ppo_checkpoint/`, `dt_v2/action_sequences.csv` (DIFFERENT from v1)

**Key insight**: DT v2's parameters will differ from v1 (transition probs changed, durations shifted, etc.). When RL retrains on the evolved shadow, its policies shift accordingly.

---

## 10) Digital Shadow Evolution Analysis
**Why:** Quantify how the shadow parameters evolved from v1 → v2

**Cells to add**
1. **Compare calibration parameters**
   - Load `dt_v1/calibration_params.json` and `dt_v2/calibration_params.json`
   - For each parameter: compute delta and % change
   - Flag significant shifts (bottleneck indicators)
   
2. **Identify process changes**
   - Which transitions decreased/increased? (activity flow changes)
   - Which activities got slower/faster? (duration shifts)
   - Which decision branches shifted? (branching probability changes)
   - Which reward function weights evolved? (priority changes)

3. **Simulator validation evolution**
   - Compare validation metrics: v1 vs v2
   - Did shadow accuracy improve with more data?
   - Which metrics changed most?

**Outputs**
- `shadow_evolution_report.json` (parameter deltas, % changes, interpretations)
- `shadow_evolution_metrics.csv` (validation metrics v1→v2)

**Checks**
- Parameter changes align with real workflow domain knowledge
- Validation accuracy improved with more data (or stayed stable)

---

## 11) Policy Interpretation & Managerial Recommendations
**Why:** Extract what each RL agent learned and translate into actionable business strategies (applied independently to v1 and v2).

**Workflow (applied once per DT version)**
1. **Load pre-computed trajectory logs (from Step 9)**
   - Read `dt_v{1,2}/action_trajectories.csv` (already saved by Step 8-9 during eval)
   - Columns: `episode_id, timestep, action, activity, queue_depth, case_age_hours, rework_count, municipality, ...`
   - No need to re-run agent — use what was already captured during training evaluation

2. **Cluster to find recurring strategies**
   - Group actions by state context (activity, queue_depth, case_age, rework_count, municipality)
   - Use K-means or hierarchical clustering to find ~5-10 action clusters per state
   - Goal: identify the core decision patterns the agent learned

3. **Extract decision rules**
   - For each cluster: fit decision tree on case features → chosen action
   - Extract interpretable rules: `IF feature_X > threshold THEN action_Y`
   - Examples:
     - "IF rework_count > 2 AND activity='Confirmation' THEN prioritize_escalation"
     - "IF queue_depth < 5 AND case_age < 7_days THEN standard_processing"
     - "IF municipality_type='Complex' THEN assign_senior_officer"

4. **Map to business language**
   - Task assignment policies: "Assign complex cases to senior staff"
   - Sequencing decisions: "Prioritize age-based cases in Review stage"
   - Resource allocation: "Concentrate workload when queue exceeds threshold"
   - Each rule gets a business interpretation + confidence score

5. **Dashboard for this version**
   - Extracted rules table (IF-THEN statements + confidence % + business translation)
   - Decision frequency heatmap (which rules fire most often?)
   - KPI impact by rule (e.g., "Rule X → improves completion time by 8%")
   - Activity/municipality breakdown of strategies
   - Interactive widget: "For Case X, what does the policy recommend?" (shows matching rules + reasoning)

**Repeat for v1 AND v2**
- Step 11a: Generate managerial dashboard for Policy v1 (from 70% DT, using `dt_v1/action_trajectories.csv`)
- Step 11b: Generate managerial dashboard for Policy v2 (from 100% DT, using `dt_v2/action_trajectories.csv`)
- Note: Comparing dashboards is a side artifact (shows how recommendations shifted), but each dashboard stands alone

**Cells to add (per version)**
1. Load trajectory CSV from `dt_v{1,2}/`
2. Normalize state features and cluster actions by state context
3. Fit decision trees and extract rules with confidence scores
4. Business language translation and mapping
5. KPI impact analysis
6. Interactive dashboard generation (HTML widget with case lookup)
7. Summary statistics (rule coverage, diversity, top-5 rules)

**Outputs (per version)**
- `dt_v{1,2}/policy_rules.csv` (extracted decision rules + confidence + business translation)
- `dt_v{1,2}/rule_impact_summary.json` (KPI deltas by rule, strategy breakdown)
- `managerial_dashboard_v{1,2}.ipynb` (interactive visualization + case recommendation widget)

**Checks**
- Rules are interpretable (≤3 conditions per rule)
- Coverage: >75% of trajectories matched by at least one rule
- Business translations validated against domain knowledge
- Dashboard is interactive and responsive
- v1 vs v2 dashboards are independently correct before any comparison

---

## 12) Reporting and Decision Gate
**Cells to add**
1. Consolidated results table (all 3 versions: v1, v2, comparison)
2. Evolution narrative: How did the workflow change?
3. Error analysis: Where does v2 policy still fail?
4. Decision: Is the shadow evolution approach validated?

**Definition of research success**
- ✓ DT v1 validated against 70% data
- ✓ DT v2 validated against 100% data (improved or stable)
- ✓ Policy v1 vs v2 differences are interpretable as workflow changes
- ✓ Managerial insights extracted and reviewed

---

## 12) Suggested notebook cell order (copy into implementation notebook)

**Phase 1: Setup & Data (Single Run)**
1. Setup/imports/config
2. Load OCEL + reusable helpers + versioning manager

**Phase 2: Digital Shadow v1 (70% Data)**
3. Graph priors extraction (v1, 70%)
4. Case-step feature engineering (v1, 70%)
5. Stats for calibration (v1, 70%)
6. Action space + reward (shared)
7. SimPy simulator (v1, 70%)
8. Simulator validation (v1, 70%)
9. Gym env wrapper (v1)
10. PPO training (v1)
11. Evaluation + action capture (v1)

**Phase 3: Digital Shadow v2 (100% Data - Evolved)**
12. Graph priors extraction (v2, 100%)
13. Case-step feature engineering (v2, 100%)
14. Stats for calibration (v2, 100%)
15. SimPy simulator (v2, 100%)
16. Simulator validation (v2, 100%)
17. Gym env wrapper (v2)
18. PPO training (v2)
19. Evaluation + action capture (v2)

**Phase 4: Evolution & Insights**
20. Shadow evolution analysis (Step 10)
21. Policy comparison & managerial dashboard (Step 11)
22. Consolidated reporting (Step 12)

---

## Definition of Done (Digital Shadow Evolution Research)

### Success Criteria

**Phase 1-2: Baseline Twin (v1)**
- ✓ Feature table and calibration artifacts persisted for v1 (70%)
- ✓ Simulator v1 validated against 70% real data
- ✓ PPO trained on v1, policy stable
- ✓ Action sequences captured

**Phase 3: Evolved Twin (v2)**
- ✓ Simulator v2 built from 100% (all) data
- ✓ Simulator v2 validated against 100% real data (accuracy maintained or improved)
- ✓ PPO trained on v2, policy converges
- ✓ Action sequences captured

**Phase 4: Research Contribution**
- ✓ **Step 10**: Shadow evolution parameters documented
  - Parameter deltas computed (v1→v2)
  - Significant shifts identified
  - Evolution report generated
- ✓ **Step 11**: Policy comparison complete
  - Rules extracted from both policies
  - Translated to business language
  - KPI differences quantified
  - Dashboard built and interactive
- ✓ **Step 12**: Research narrative complete
  - How did workflow change?
  - How did RL policy adapt?
  - What are the managerial implications?
  - Papers cited: Kritzinger et al., Tao et al., Seipolt et al.

### Research Validation

- ✓ DT v1 and v2 are both valid (validated against real data)
- ✓ Parameter changes between v1 and v2 are interpretable
- ✓ Policy differences between v1 and v2 align with shadow evolution
- ✓ Extracted rules are human-understandable (2-3 conditions max)
- ✓ Dashboard is interactive and actionable

## **CQL (Conservative Q-Learning) - Detailed Overview**

**What is CQL?**
- **Not** a pre-trained model—it's an **offline RL algorithm** that learns from your historical municipal data (logs)
- Published 2020 by Kumar et al. as a solution to the "offline RL problem"
- Available in Stable-Baselines3 via `sb3-contrib`

---

## **How CQL Works (Simple Explanation)**



In [ ]:
Traditional RL:  Agent explores → gets rewards → learns
Offline RL (CQL): Agent learns ONLY from historical data → stays conservative



CQL adds a **pessimistic penalty** to Q-learning:
1. Learns Q-values from your municipal logs (like your BC does, but via RL)
2. **Key difference**: Penalizes actions it hasn't seen in the data heavily
3. Result: Policy stays close to observed expert behavior (safe) but optimizes where data allows

**Why CQL for this comparison:**
- Learns from same historical municipal data as your BC
- Can optimize beyond pure imitation (unlike BC)
- Stays safe because it's conservative (won't break constraints badly)
- Direct competitor to BC approach

---

## **Training Time Requirements**

| Component | Time | Notes |
|---|---|---|
| **BC Model** | 5-15 min | Fast (supervised learning) |
| **CQL Training** | 30-120 min | Slower (RL exploration on logged data) |
| **Evaluation (both)** | 10-20 min | Rollouts on test cases |
| **Total Comparison** | **1.5-3 hours** | Depending on data size & hyperparams |

**Factors affecting CQL speed:**
- Number of historical cases (more = slower)
- Action space size (resource allocation dimensions)
- Episode length (steps to process each case)
- Network architecture complexity

---

## **Implementation Approach**

Here's the workflow:

### **Step 1: Prepare Offline Dataset**


In [ ]:
# Convert your municipal logs to RL format
observations = []  # states from cases
actions = []       # resource assignments (what actually happened)
rewards = []       # KPI improvements (duration_reduced, etc)
terminals = []     # case completion

# Package into OfflineDataset



### **Step 2: Train CQL**


In [ ]:
from sb3_contrib import CQL

# Use your historical data
cql_model = CQL(
    "MlpPolicy",
    your_env,
    learning_rate=1e-4,
    buffer_size=len(offline_data),  # Fixed size (no new data)
    batch_size=256,
)

cql_model.learn(total_timesteps=100_000)  # Much shorter than PPO



### **Step 3: Compare Against BC**


In [ ]:
metrics = {
    'BC_duration': evaluate_bc_model(bc_policy),
    'CQL_duration': evaluate_cql_model(cql_model),
    'FCFS_duration': evaluate_fcfs_baseline(),
    'Greedy_duration': evaluate_greedy_baseline(),
}



---

## **CQL vs BC: Key Differences**

| Aspect | BC (Your Current) | CQL |
|---|---|---|
| **Training data** | Historical actions (what happened) | States, actions, rewards (why it happened) |
| **Optimization** | Minimize action prediction error | Maximize expected reward from data |
| **Risk** | Conservative (copies exactly) | Slightly more aggressive (optimizes within data support) |
| **When it's better** | Data is high-quality expert | Data has suboptimal decisions you can improve on |
| **When it fails** | Data distribution shifts | Too conservative, stays at expert level |

---

## **Practical Timeline for Your Comparison**

**Assuming BPIC dataset (5 municipalities, ~10k cases total):**



In [ ]:
Monday:
  - 10 min: Prepare offline RL dataset from your logs
  - 20 min: Set up CQL environment wrapper
  - 90 min: Train CQL (overnight if you want)

Tuesday:
  - 30 min: Run evaluation on test set (compare BC vs CQL vs FCFS)
  - 20 min: Generate comparison plots/tables
  - 10 min: Write results summary
  
Total: 3-4 hours wall-clock time (mostly training)



**Can you speed it up?**

- ✅ **Reduce offline data**: Sample 3000 cases instead of 10k → 50% faster training
- ✅ **Smaller network**: 64-dims instead of 256-dims → 30% faster  
- ✅ **Fewer timesteps**: 50k instead of 100k → 50% faster (less convergence)
- ❌ **GPU won't help much**: CQL is memory-bound, not compute-bound

---

## **My Recommendation: Do This Comparison**

**Reasons:**
1. **Complementary to BC**: Shows if RL optimization beats pure imitation
2. **Quick to implement**: ~2 hours total effort (you have the infrastructure)
3. **Strong publication storyline**: 
   - *"BC learns what experts do, CQL learns what optimizes rewards—but BC wins because constraints matter more than pure metric optimization in municipal settings"* (or vice versa)
4. **Reuses your data**: Your calibration data becomes the offline dataset

**Realistic outcome:**
- CQL probably performs **similar or slightly worse than BC** in bureaucratic workflows (because expert decisions are already pretty good, and CQL gets conservative)
- BC probably performs **much better than FCFS/Greedy** (validates your approach)

---

## **Quick Implementation: Want Me To Add CQL To Your Notebook?**

I can add a comparison cell that:
1. ✅ Converts your municipal logs to CQL offline dataset  
2. ✅ Trains CQL model in ~60-90 min
3. ✅ Evaluates both BC and CQL on same test cases
4. ✅ Generates side-by-side metrics table

Would you like me to implement this? It would be a new section in your step8_step9_colab_combined.ipynb that runs after your PPO training completes.